# ImmunoClassifier — PBMC Classification Benchmark

> **ImmunoClassifier v0.1.0**
> Author: Patrick Grady
> Anthropic Claude Opus 4.6 used for code formatting and cleanup assistance.
> License: MIT License — See LICENSE

**End-to-end demo**: synthetic PBMC data → preprocessing → train Logistic & XGBoost → evaluate → visualize

This notebook is fully self-contained — no data downloads, no GPU, runs in ~30 seconds.

---

### What this notebook demonstrates

| Step | API used |
|------|----------|
| Generate realistic immune cell data | `IMMUNE_MARKERS` registry |
| QC / normalize / HVG / PCA / UMAP | `preprocess()` pipeline |
| Baseline classification | `LogisticClassifier` |
| Gradient-boosted classification | `XGBoostClassifier` |
| Per-class & rare-cell evaluation | `evaluate_predictions()`, `rare_cell_analysis()` |
| Confusion matrices & benchmark plots | `plot_confusion_matrix()`, `plot_benchmark_comparison()` |
| UMAP predicted vs ground truth | `plot_umap_predictions()` |
| Feature importance (gene ranking) | `XGBoostClassifier.get_feature_importance()` |
| Model persistence | `save()` / `load()` round-trip |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import tempfile, os, warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, fontsize=12)

print("Imports OK")

## 1. Generate Synthetic PBMC Data

We create ~2,000 cells across **8 immune cell types** with biologically realistic
marker gene expression patterns.  Rare types (pDC, Treg) are deliberately
under-represented to mirror real immune datasets.

The marker genes come directly from the `IMMUNE_MARKERS` registry in the
ImmunoClassifier package — the same markers used for downstream evaluation.

In [ ]:
from immunoclassifier.data.datasets import IMMUNE_MARKERS

np.random.seed(42)

# Cell types and realistic proportions (mirrors a typical PBMC sample)
CELL_TYPES = {
    "CD4+ T cells":       400,   # ~20%
    "CD8+ T cells":       300,   # ~15%
    "B cells":            250,   # ~12%
    "NK cells":           200,   # ~10%
    "Classical Monocyte": 350,   # ~18%
    "cDC2":               150,   #  ~8%
    "Treg":                60,   #  ~3%  ← rare
    "pDC":                 40,   #  ~2%  ← rare
}

n_total = sum(CELL_TYPES.values())  # 1,750
n_background_genes = 2000

# Collect all unique marker genes across included types
marker_set = set()
for ct in CELL_TYPES:
    if ct in IMMUNE_MARKERS:
        marker_set.update(IMMUNE_MARKERS[ct])
marker_genes = sorted(marker_set)

# Background gene names
background_genes = [f"BKG_{i}" for i in range(n_background_genes)]
all_genes = marker_genes + background_genes
n_genes = len(all_genes)

print(f"Generating {n_total} cells, {n_genes} genes ({len(marker_genes)} marker + {n_background_genes} background)")
print(f"Cell types: {list(CELL_TYPES.keys())}")

In [ ]:
# Build the count matrix: marker genes get elevated expression in their cell type
X = np.random.poisson(lam=0.5, size=(n_total, n_genes)).astype(np.float32)

labels = []
row = 0
for ct, n_cells in CELL_TYPES.items():
    labels.extend([ct] * n_cells)
    if ct in IMMUNE_MARKERS:
        for gene in IMMUNE_MARKERS[ct]:
            if gene in marker_genes:
                col = all_genes.index(gene)
                # Signal: high expression in the correct cell type
                X[row:row + n_cells, col] = np.random.poisson(lam=8.0, size=n_cells)
    row += n_cells

# Build AnnData
adata = ad.AnnData(
    X=csr_matrix(X),
    obs=pd.DataFrame({"cell_type": labels}, index=[f"cell_{i}" for i in range(n_total)]),
)
adata.var_names = all_genes
# Mark mitochondrial genes (none here — just set the flag so QC works)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.layers["counts"] = adata.X.copy()

print(adata)
print(f"\nLabel distribution:\n{adata.obs['cell_type'].value_counts().to_string()}")

## 2. Preprocess

Run the full ImmunoClassifier preprocessing pipeline:
**QC filtering → normalization → HVG selection → PCA → neighbor graph → UMAP → Leiden clustering**

In [ ]:
from immunoclassifier.data.preprocessing import preprocess

adata = preprocess(
    adata,
    min_genes=10,        # lower threshold for synthetic data
    min_cells=1,
    max_pct_mito=100,    # no mito genes in synthetic data
    n_top_genes=500,     # smaller HVG set for speed
    n_pcs=30,
    n_neighbors=15,
    copy=True,
)

print(f"After preprocessing: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")

In [ ]:
# Visualize the ground-truth labels on the UMAP
sc.pl.umap(adata, color="cell_type", title="Ground Truth Cell Types", frameon=False)

## 3. Train & Compare Models

We train two classifiers on the same preprocessed data:

| Model | Why include it |
|-------|----------------|
| **Logistic Regression** | Fast baseline; surprisingly strong on HVG expression |
| **XGBoost** | Handles class imbalance, gives gene-level feature importance |

In [ ]:
from immunoclassifier.models.logistic import LogisticClassifier
from immunoclassifier.models.xgboost_model import XGBoostClassifier

# --- Logistic Regression ---
logistic = LogisticClassifier(C=1.0, max_iter=500)
logistic_metrics = logistic.train(adata, label_key="cell_type", val_fraction=0.15)

print("\n=== Logistic Regression ===")
for k, v in logistic_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# --- XGBoost ---
xgb = XGBoostClassifier(n_estimators=100, max_depth=6, learning_rate=0.1)
xgb_metrics = xgb.train(adata, label_key="cell_type", val_fraction=0.15, early_stopping_rounds=10)

print("\n=== XGBoost ===")
for k, v in xgb_metrics.items():
    if k != "evals_result":
        print(f"  {k}: {v}")

## 4. Evaluate — Full Metrics & Rare-Cell Analysis

Use `evaluate_predictions()` for overall + per-class metrics, then
`rare_cell_analysis()` to compare performance on rare vs abundant subtypes.

In [ ]:
from immunoclassifier.evaluation.metrics import evaluate_predictions, rare_cell_analysis

y_true = adata.obs["cell_type"].values

# Predictions
y_pred_logistic = logistic.predict(adata)
y_pred_xgb = xgb.predict(adata)

# Full evaluation
eval_logistic = evaluate_predictions(y_true, y_pred_logistic)
eval_xgb = evaluate_predictions(y_true, y_pred_xgb)

# Summary
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced Accuracy", "Macro F1", "Weighted F1", "Cohen's κ"],
    "Logistic": [
        eval_logistic["accuracy"],
        eval_logistic["balanced_accuracy"],
        eval_logistic["macro_f1"],
        eval_logistic["weighted_f1"],
        eval_logistic["cohen_kappa"],
    ],
    "XGBoost": [
        eval_xgb["accuracy"],
        eval_xgb["balanced_accuracy"],
        eval_xgb["macro_f1"],
        eval_xgb["weighted_f1"],
        eval_xgb["cohen_kappa"],
    ],
}).set_index("Metric")

print(summary.to_string(float_format="{:.4f}".format))

In [ ]:
# Rare-cell analysis (Treg = 60 cells, pDC = 40 cells — both below threshold of 100)
rare_logistic = rare_cell_analysis(y_true, y_pred_logistic, threshold=100)
rare_xgb = rare_cell_analysis(y_true, y_pred_xgb, threshold=100)

print("=== Rare Cell Performance (< 100 cells) ===")
print(f"  Rare types: {rare_xgb['rare_types']}")
print(f"  Logistic — Rare acc: {rare_logistic['rare_accuracy']:.4f}, F1: {rare_logistic['rare_macro_f1']:.4f}")
print(f"  XGBoost  — Rare acc: {rare_xgb['rare_accuracy']:.4f}, F1: {rare_xgb['rare_macro_f1']:.4f}")
print(f"\n  Abundant types: {rare_xgb['abundant_types']}")
print(f"  Logistic — Abundant acc: {rare_logistic['abundant_accuracy']:.4f}")
print(f"  XGBoost  — Abundant acc: {rare_xgb['abundant_accuracy']:.4f}")

In [ ]:
# Per-class breakdown (XGBoost)
print("=== Per-Class Metrics (XGBoost) ===")
print(eval_xgb["per_class"].to_string(float_format="{:.3f}".format))

## 5. Visualize

### 5a. Confusion Matrices

In [ ]:
from immunoclassifier.evaluation.plots import (
    plot_confusion_matrix,
    plot_benchmark_comparison,
    plot_umap_predictions,
)

fig = plot_confusion_matrix(
    y_true, y_pred_logistic,
    normalize=True,
    title="Logistic Regression — Normalized Confusion Matrix",
    figsize=(10, 8),
)
plt.show()

In [ ]:
fig = plot_confusion_matrix(
    y_true, y_pred_xgb,
    normalize=True,
    title="XGBoost — Normalized Confusion Matrix",
    figsize=(10, 8),
)
plt.show()

### 5b. Benchmark Comparison Chart

In [ ]:
benchmark_results = {
    "Logistic Regression": eval_logistic,
    "XGBoost": eval_xgb,
}

fig = plot_benchmark_comparison(
    benchmark_results,
    metrics=["accuracy", "balanced_accuracy", "macro_f1", "cohen_kappa"],
    figsize=(9, 5),
)
plt.show()

### 5c. XGBoost Feature Importance — Top Marker Genes

In [ ]:
importance = xgb.get_feature_importance(importance_type="gain", top_n=20)

fig, ax = plt.subplots(figsize=(8, 6))
genes = list(importance.keys())
scores = list(importance.values())

# Color marker genes differently from background genes
colors = ["#2196F3" if not g.startswith("BKG_") else "#BDBDBD" for g in genes]

ax.barh(range(len(genes)), scores, color=colors)
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes)
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("XGBoost Top 20 Genes — Known Markers in Blue")
plt.tight_layout()
plt.show()

# How many of the top genes are known markers?
n_marker_in_top = sum(1 for g in genes if not g.startswith("BKG_"))
print(f"\n{n_marker_in_top}/{len(genes)} of top-ranked genes are known immune markers")

### 5d. UMAP — Predicted vs Ground Truth

In [ ]:
# Store predictions in adata.obs for UMAP plotting
adata.obs["pred_logistic"] = y_pred_logistic
adata.obs["pred_xgboost"] = y_pred_xgb

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False, title="Ground Truth", frameon=False)
sc.pl.umap(adata, color="pred_logistic", ax=axes[1], show=False, title="Logistic Predictions", frameon=False)
sc.pl.umap(adata, color="pred_xgboost", ax=axes[2], show=False, title="XGBoost Predictions", frameon=False)
plt.tight_layout()
plt.show()

### 5e. Confidence Scores

In [ ]:
# Predict with confidence using the base class method
preds, confidences = logistic.predict_with_confidence(adata)
adata.obs["confidence"] = confidences

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# UMAP colored by confidence
sc.pl.umap(adata, color="confidence", ax=axes[0], show=False,
           title="Prediction Confidence", frameon=False, cmap="RdYlGn",
           vmin=0, vmax=1)

# Distribution per cell type
import seaborn as sns
conf_df = adata.obs[["cell_type", "confidence"]].copy()
order = conf_df.groupby("cell_type")["confidence"].median().sort_values().index
sns.boxplot(data=conf_df, x="confidence", y="cell_type", order=order, ax=axes[1],
            palette="viridis")
axes[1].set_title("Confidence by Cell Type")
axes[1].set_xlabel("Max Probability")
plt.tight_layout()
plt.show()

# Low-confidence cells are candidates for manual review
low_conf = (confidences < 0.5).sum()
print(f"Cells with confidence < 0.5: {low_conf}/{len(confidences)} ({100*low_conf/len(confidences):.1f}%)")

## 6. Model Persistence — Save / Load Round-Trip

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    # Save both models
    logistic_path = os.path.join(tmpdir, "logistic_model.pkl")
    xgb_path = os.path.join(tmpdir, "xgb_model")

    logistic.save(logistic_path)
    xgb.save(xgb_path)

    # Show saved files
    saved_files = os.listdir(tmpdir)
    print(f"Saved files: {saved_files}")
    for f in saved_files:
        size = os.path.getsize(os.path.join(tmpdir, f))
        print(f"  {f}: {size:,} bytes")

    # Load into fresh instances
    logistic_loaded = LogisticClassifier()
    logistic_loaded.load(logistic_path)

    xgb_loaded = XGBoostClassifier()
    xgb_loaded.load(xgb_path)

    # Verify predictions match
    y_reload_logistic = logistic_loaded.predict(adata)
    y_reload_xgb = xgb_loaded.predict(adata)

    assert np.array_equal(y_pred_logistic, y_reload_logistic), "Logistic predictions differ after reload!"
    assert np.array_equal(y_pred_xgb, y_reload_xgb), "XGBoost predictions differ after reload!"

    print("\n✓ Save/load round-trip verified — predictions identical.")

## Summary

This notebook exercised the **complete ImmunoClassifier API**:

- `data.datasets.IMMUNE_MARKERS` — biologically curated marker gene registry
- `data.preprocessing.preprocess()` — full QC → UMAP pipeline
- `models.LogisticClassifier` / `XGBoostClassifier` — train / predict / save / load
- `BaseClassifier.predict_with_confidence()` — probability-based confidence scoring
- `evaluation.metrics` — accuracy, balanced accuracy, macro F1, Cohen's κ, per-class, rare-cell
- `evaluation.plots` — confusion matrix, benchmark comparison, UMAP overlay

### Next steps

1. **Real data**: Replace the synthetic generator with `load_pbmc_10k()` to classify real PBMCs
2. **More models**: Add `ScVIClassifier` (VAE-based) and `GNNClassifier` (graph attention) to the benchmark
3. **Cross-dataset**: Train on PBMC 10k → evaluate on Tabula Sapiens immune compartment
4. **Hyperparameter tuning**: Use `immunoclassifier.training.hyperopt` for Optuna-driven search

---

*ImmunoClassifier v0.1.0 — Any usage is subject to this software's license.*